In [25]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

This notebook creates the growth allocation list from the local growth assignments. The path to the log is here: "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log.xlsx" this should be updated anytime new growth is reported and used as a reference and read in here. Images and lat/lons are here: "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log_Residential.docx", "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log_NonResidential.docx".    

Process: took the input from our locals (name and date recorded in spreadsheet) and found them on the map. Drew and then digitized a polygon and GIS analyst returns a list of Census blocks with the percent they are covered by these polygons. In this notebook we'll take the total growth reported and allocate it by percent to these blocks.

In [26]:
df = pd.read_excel(r"G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log.xlsx", sheet_name="Non-Residential", usecols=["ID", "Number of Jobs Modeled"])

In [27]:
df

,ID,Number of Jobs Modeled
0,NR1,150
1,NR2,25
2,NR3,25
3,NR4,45
4,NR5,35
5,NR6,35
6,NR7,20


In [28]:
df_perc = pd.read_excel(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\2-interim/StudyArea_Censusblocksummary.xls")
df_perc.head(2)

,OBJECTID,ID,GEOID,PCT_INTERSECT,PCT_BLOCK,SIZE_BLOCK
0,1,NR1,210472013011038,0.173787,0.502024,97.549293
1,2,NR1,210472013011036,22.136406,99.497971,151.783406


Going to take the liberty of saying that if a block is less than 50% covered by the polygon let's just not... the logic here is that ultimately the allocation script takes the locally reported employment as the ground truth and so we don't want to replace what's there when the polygon might only barely touch the space. The assumption is that if the polygon covers most or all of the space that the reported activity is the activity at that block (this holds true as these are mostly like... industrial developments in mostly rural counties instead of dense urban areas) - also with the nature of this being a general allocation it's okay if the residential or employment is a little more concentrated literally next door. The 50% is also based on the spread.. there are only like 4 blocks between 25% and 50% so let's just drop 'em.

In [29]:
# Identify NR developments
nr_mask = df_perc['ID'].str.contains('NR', na=False)

# Identify IDs with at least one block meeting the 50% threshold
valid_ids = df_perc.loc[
    nr_mask & (df_perc['PCT_BLOCK'] >= 50), 'ID'
].unique()

# Keep blocks meeting threshold OR all blocks for IDs with no qualifying blocks
df_perc_filtered = df_perc.loc[
    nr_mask &
    (
        (df_perc['PCT_BLOCK'] >= 50) |
        (~df_perc['ID'].isin(valid_ids))
    )
].copy()

# Re-normalize percentages to sum to 100% per ID
df_perc_filtered['PCT_INTERSECT_NORM'] = (
    df_perc_filtered['PCT_INTERSECT']
    / df_perc_filtered.groupby('ID')['PCT_INTERSECT'].transform('sum')
    * 100
)

# Match employment totals
df_perc_filtered['EMP_TOTAL'] = df_perc_filtered['ID'].map(
    df.set_index('ID')['Number of Jobs Modeled']
)

# Allocate employment
df_perc_filtered['EMP_PER_BLOCK'] = (
    df_perc_filtered['EMP_TOTAL']
    * (df_perc_filtered['PCT_INTERSECT_NORM'] / 100)
)

In [30]:
df_perc_filtered.sort_values(by = 'PCT_BLOCK', ascending = True).head(10)

,OBJECTID,ID,GEOID,PCT_INTERSECT,PCT_BLOCK,SIZE_BLOCK,PCT_INTERSECT_NORM,EMP_TOTAL,EMP_PER_BLOCK
1,2,NR1,210472013011036,22.136406,99.497971,151.783406,100.0,150,150.0
24,25,NR6,212219702022008,43.172145,99.981416,99.588439,100.0,35,35.0
21,22,NR5,210472013041026,1.204076,100.000000,117.810120,100.0,35,35.0
2,3,NR4,210472014002075,4.853372,100.000000,95.688894,100.0,45,45.0
22,23,NR2,210472013041026,0.118433,100.000000,117.810120,100.0,25,25.0
23,24,NR7,212219702011014,12.909711,100.000000,147.538474,100.0,20,20.0


In [31]:
df_perc_filtered = df_perc_filtered.rename(columns = {'GEOID': 'GEO_ID'})
df_perc_filtered['GEO_ID'] = df_perc_filtered['GEO_ID'].astype(str)

In [32]:
df.head()

,ID,Number of Jobs Modeled
0,NR1,150
1,NR2,25
2,NR3,25
3,NR4,45
4,NR5,35


In [33]:
# Match total employment from development dataframe
df_perc_filtered['EMP_TOTAL'] = df_perc_filtered['ID'].map(
    df.set_index('ID')['Number of Jobs Modeled']
)

# Allocate employment based on percentage of development polygon in each block
df_perc_filtered['EMP_PER_BLOCK'] = (
    df_perc_filtered['EMP_TOTAL'] * (df_perc_filtered['PCT_INTERSECT'] / 100)
)

In [34]:
allocation_check = df_perc_filtered.groupby('ID').agg(
    PCT_TOTAL=('PCT_INTERSECT_NORM', 'sum'),
    EMP_TOTAL=('EMP_TOTAL', 'first'),
    EMP_ALLOCATED=('EMP_PER_BLOCK', 'sum')
)

allocation_check['EMP_DIFF'] = (
    allocation_check['EMP_TOTAL'] - allocation_check['EMP_ALLOCATED']
)

print(allocation_check)

     PCT_TOTAL  EMP_TOTAL  EMP_ALLOCATED    EMP_DIFF
ID                                                  
NR1      100.0        150      33.204609  116.795391
NR2      100.0         25       0.029608   24.970392
NR4      100.0         45       2.184017   42.815983
NR5      100.0         35       0.421427   34.578573
NR6      100.0         35      15.110251   19.889749
NR7      100.0         20       2.581942   17.418058


In [35]:
missing_ids = df.loc[
    ~df['ID'].isin(df_perc_filtered['ID'])
]

print(missing_ids)

    ID  Number of Jobs Modeled
2  NR3                      25
